In [1]:
!pip install vllm bitsandbytes pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 93.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 88.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 25.4 MB/s eta 0:00:00
   ━━━━

In [2]:
import os, subprocess

result = subprocess.run(["find", "/usr", "-name", "libcuda.so*"], capture_output=True, text=True)
print(result.stdout)

/usr/local/nvidia/lib64/libcuda.so.1
/usr/local/nvidia/lib64/libcuda.so.580.105.08
/usr/local/nvidia/lib64/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.1
/usr/local/cuda-12.8/compat/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.570.124.06



In [3]:
os.environ["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# clear the flashinfer JIT cache so it recompiles with the correct paths
!rm -rf /root/.cache/flashinfer/

In [6]:
server = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--dtype", "float16",
    "--tensor-parallel-size", "2",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.90",
    "--enable-chunked-prefill",
    "--max-num-batched-tokens", "512",
    "--cudagraph-capture-sizes", "1", "2", "4", "8", "16",
    "--port", "8000",
],
    env={**os.environ},
    stdout=open("/kaggle/working/vllm.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Server PID:", server.pid)
with open("/kaggle/working/vllm.pid", "w") as f:
    f.write(str(server.pid))

Server PID: 195


In [7]:
import time, subprocess

print("Waiting for server...")
for _ in range(120):
    time.sleep(60)
    result = subprocess.run(["grep", "-c", "Application startup complete", "/kaggle/working/vllm.log"],
                            capture_output=True, text=True)
    if result.stdout.strip() == "1":
        print("*** SERVER READY ***")
        break
    # show last line of log so you can see progress
    tail = subprocess.run(["tail", "-1", "/kaggle/working/vllm.log"], capture_output=True, text=True)
    print(tail.stdout.strip())
else:
    print("Timed out — check /kaggle/working/vllm.log")

Waiting for server...
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
(Worker_TP0 pid=274) INFO 04-01 19:34:46 [cuda.py:317] Using FLASHINFER attention backend out of potential backends: ['FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(Worker_TP0 pid=274) 
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:15<00:15,  7.82s/it]
(Worker_TP0 pid=274) INFO 04-01 19:37:20 [monitor.py:76] Initial profiling/warmup run took 0.82 s
(EngineCore pid=247) INFO 04-01 19:37:50 [shm_broadcast.py:681] No available shared memory broadcast block found in 60 seconds. This typically happens when some processes are hanging or doing some time-consuming work (e.g. compilation, weight/kv cache quantization).
*** SERVER READY ***


In [10]:
from pyngrok import ngrok

ngrok.set_auth_token("3AVBps8CZBIFHhuOziZytnu1rPF_6kjruq6p7y1vkWYcUwywE")
tunnel = ngrok.connect(8000)
print("PUBLIC URL:", tunnel.public_url)

PUBLIC URL: https://rufflike-heteromorphic-lacie.ngrok-free.dev


In [9]:
ngrok.kill()